# Executive Summary

This project analyzes over 28,000 Amazon customer reviews to understand customer sentiment, identify dissatisfaction drivers, and generate actionable business insights.

Using Natural Language Processing (NLP) and Machine Learning techniques, customer reviews were transformed into structured sentiment signals that can support product improvement, customer experience optimization, and strategic decision-making.

Key objectives:

- Understand customer sentiment distribution
- Identify positive and negative customer feedback themes
- Build predictive sentiment classification models
- Compare machine learning approaches
- Generate business recommendations

# Business Problem

Customer reviews contain valuable insights regarding product quality, usability, pricing, and customer satisfaction.

However, manually reading thousands of reviews is impractical.

The objective is to develop an automated sentiment analysis pipeline capable of:

- Classifying customer sentiment
- Detecting dissatisfaction patterns
- Understanding category-level sentiment
- Supporting business decision-making through data-driven insights

# IMPORTING LIBRARIES

In [ ]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Feature Engineering
from sklearn.feature_extraction.text import TfidfVectorizer

# Modeling
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Word Cloud
from wordcloud import WordCloud

# Warnings
import warnings
warnings.filterwarnings('ignore')

# LOADING DATASET

In [ ]:
df = pd.read_csv("Dataset.csv")

In [ ]:
df.shape

# OVERVIEW

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# DATA ASSESSMENT

In [ ]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10,5))
missing_values[missing_values > 0].sort_values().plot(kind='barh')
plt.title("Missing Values")
plt.show()

# SENTIMENT LABEL CREATION

In [ ]:
def create_sentiment(rating):

    if rating >= 4:
        return "Positive"

    elif rating == 3:
        return "Neutral"

    else:
        return "Negative"

In [ ]:
df["sentiment"] = df["reviews.rating"].apply(create_sentiment)

In [ ]:
df["sentiment"].value_counts()

# Executive KPI Summary

This section provides a high-level overview of customer sentiment across all Amazon reviews.

In [ ]:
total_reviews = len(df)

positive_pct = (
    (df["sentiment"] == "Positive")
    .mean() * 100
)

negative_pct = (
    (df["sentiment"] == "Negative")
    .mean() * 100
)

neutral_pct = (
    (df["sentiment"] == "Neutral")
    .mean() * 100
)

print(f"Total Reviews: {total_reviews:,}")
print(f"Positive Reviews: {positive_pct:.2f}%")
print(f"Negative Reviews: {negative_pct:.2f}%")
print(f"Neutral Reviews: {neutral_pct:.2f}%")

## Class Imbalance Note

The Neutral class (rating = 3) represents a very small proportion
of the dataset. As a result, models may show low F1 scores for
this class — this is expected, not an error.

In a production setting, options include:
- Merging Neutral into Positive/Negative (binary classification)
- Collecting more neutral-rated reviews to balance the class
- Using oversampling techniques (e.g. SMOTE) on the minority class

In [ ]:
print("Class distribution (%):")
print((df["sentiment"].value_counts(normalize=True)*100).round(2))

# SENTIMENT DISTRIBUTION ANALYSIS

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x='sentiment'
)

plt.title("Sentiment Distribution")
plt.show()

# PRODUCT CATEGORY SENTIMENT ANALYSIS

In [ ]:
category_sentiment = pd.crosstab(
    df['primaryCategories'],
    df['sentiment']
)

In [ ]:
category_sentiment

In [ ]:
ax = category_sentiment.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)
ax.set_title("Sentiment by Product Category", fontsize=14)
ax.set_xlabel("Product Category")
ax.set_ylabel("Number of Reviews")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
positive_reviews = df[
    df["sentiment"] == "Positive"
]

positive_categories = (

    positive_reviews

    .groupby("primaryCategories")

    .size()

    .sort_values(ascending=False)

)

positive_categories.head(10)

In [ ]:
plt.figure(figsize=(10,5))

positive_categories.head(10).plot(
    kind='bar'
)

plt.title(
    "Categories Receiving Most Positive Reviews"
)

plt.show()

In [ ]:
negative_reviews = df[
    df["sentiment"] == "Negative"
]

negative_categories = (

    negative_reviews

    .groupby("primaryCategories")

    .size()

    .sort_values(ascending=False)

)

negative_categories.head(10)

In [ ]:
plt.figure(figsize=(10,5))

negative_categories.head(10).plot(
    kind='bar'
)

plt.title(
    "Categories Receiving Most Negative Reviews"
)

plt.show()

# TEXT CLEANING

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

In [ ]:
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r"http\S+","",text)

    text = re.sub(r"[^a-zA-Z\s]","",text)

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

In [ ]:
df["clean_review"] = df["reviews.text"].apply(clean_text)

In [ ]:
df["review_length"] = df["reviews.text"].apply(
    lambda x: len(str(x).split())
)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="sentiment", y="review_length",
            order=["Positive", "Neutral", "Negative"])
plt.title("Review Length by Sentiment", fontsize=14)
plt.ylabel("Word Count")
plt.xlabel("Sentiment")
plt.ylim(0, 300)
plt.tight_layout()
plt.show()

print(df.groupby("sentiment")["review_length"].median())

# WORDCLOUD ANALYSIS 

In [ ]:
positive_text = " ".join(
    df[df["sentiment"]=="Positive"]
    ["clean_review"]
)

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400
).generate(positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Common Words in Positive Reviews", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
negative_text = " ".join(
    df[df["sentiment"]=="Negative"]
    ["clean_review"]
)

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400
).generate(negative_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Common Words in Negative Reviews", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
neutral_text = " ".join(
    df[df["sentiment"]=="Neutral"]
    ["clean_review"]
)

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400
).generate(neutral_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Common Words in Neutral Reviews", fontsize=14)
plt.tight_layout()
plt.show()

# N-Gram Analysis

Word clouds show individual words.

N-Grams help identify common customer phrases.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bigram = CountVectorizer(
    ngram_range=(2,2),
    stop_words='english',
    max_features=20
)

X_bigram = bigram.fit_transform(
    df["clean_review"]
)

bigram_freq = pd.DataFrame({

    "Phrase": bigram.get_feature_names_out(),

    "Frequency":
    X_bigram.sum(axis=0).A1

})

bigram_freq.sort_values(
    "Frequency",
    ascending=False
).head(15)

# TF-IDF Feature

In [ ]:
y = df["sentiment"]

In [ ]:
X_text = df["clean_review"]

In [ ]:
X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(X_text_train)   
X_test  = tfidf.transform(X_text_test) 

Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')

lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

In [ ]:
accuracy_score(y_test,pred_lr)

In [ ]:
print(classification_report(
    y_test,
    pred_lr
))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(

    lr,

    X_train,

    y_train,

    cv=5,

    scoring='f1_weighted'

)

print(
    "Average CV Score:",
    cv_scores.mean()
)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {

    'C':[0.01,0.1,1,10]

}

grid = GridSearchCV(

    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    param_grid,

    cv=5,

    scoring='f1_weighted'

)

grid.fit(
    X_train,
    y_train
)

print(grid.best_params_)

In [ ]:
lr = grid.best_estimator_

 Naive Bayes

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

nb = MultinomialNB()
sample_weights = compute_sample_weight('balanced', y_train)
nb.fit(X_train, y_train, sample_weight=sample_weights)
pred_nb = nb.predict(X_test)

In [ ]:
print("=== Naive Bayes ===")
print(classification_report(y_test, pred_nb))

Random Forest 

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train,y_train)

pred_rf = rf.predict(X_test)

In [ ]:
print("=== Random Forest ===")
print(classification_report(y_test, pred_rf))

Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

models_preds = [
    ("Logistic Regression", pred_lr),
    ("Naive Bayes",         pred_nb),
    ("Random Forest",       pred_rf)
]

labels = sorted(y_test.unique())

for ax, (name, pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=labels, yticklabels=labels,
                cmap='Blues')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, pred_lr),
                 accuracy_score(y_test, pred_nb),
                 accuracy_score(y_test, pred_rf)],
    "Precision": [precision_score(y_test, pred_lr, average='weighted'),
                  precision_score(y_test, pred_nb, average='weighted'),
                  precision_score(y_test, pred_rf, average='weighted')],
    "Recall":    [recall_score(y_test, pred_lr, average='weighted'),
                  recall_score(y_test, pred_nb, average='weighted'),
                  recall_score(y_test, pred_rf, average='weighted')],
    "F1 Score":  [f1_score(y_test, pred_lr, average='weighted'),
                  f1_score(y_test, pred_nb, average='weighted'),
                  f1_score(y_test, pred_rf, average='weighted')]
})

results = results.sort_values(by="F1 Score", ascending=False).round(4)
results

In [ ]:
feature_names = np.array(
    tfidf.get_feature_names_out()
)

positive_terms = feature_names[
    np.argsort(
        lr.coef_[0]
    )[-15:]
]

negative_terms = feature_names[
    np.argsort(
        lr.coef_[0]
    )[:15]
]

print("Positive Terms")
print(positive_terms)

print("\nNegative Terms")
print(negative_terms)

- Prioritize products with highest negative sentiment rates.

- Investigate recurring complaint keywords.

- Monitor sentiment trends monthly.

- Integrate sentiment monitoring into customer experience dashboards.

- Use review sentiment as an early-warning system for product quality issues.

## Conclusion & Best Model

Based on F1 Score comparison across all three models:

- **Random Forest** delivered the best balance of accuracy 
  and interpretability, making it the recommended model for production.
- Naive Bayes was fastest to train but showed weaker performance 
  on the minority Negative class.
- Random Forest achieved the highest F1 score across all three models,
  demonstrating strong performance on both majority and minority classes.

**Recommended model: Random Forest**

In [ ]:
print("=" * 40)
print("FINAL MODEL SUMMARY")
print("=" * 40)
print(results.to_string(index=False))